# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset title and description
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All Croissant entities are referenced by their `@id`.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.metadata.get_record_sets()
print("Record Sets available in this dataset (by @id):\n")
for rs in record_sets:
    print(f"  • RecordSet @id: {rs['@id']}  |  name: {rs.get('name','')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    if fields:
        print("    Fields:")
        for field in fields:
            # Field could be an @id mapping or dict
            fid = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"      - Field @id: {fid}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use only record set and field `@id`s from the overview.

In [ ]:
# Select the major record set by @id
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/78d7e0b4-1abc-487d-a045-7ee725313af7'  # Replace with main @id from overview step

record_sets_ids = [rs['@id'] for rs in dataset.metadata.get_record_sets()]
print("RecordSet @id list:")
for rsid in record_sets_ids:
    print(rsid)

# Load records for all record sets into DataFrames
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
    else:
        print(f"No records extracted for record set: {record_set_id}")

if main_record_set_id in dataframes:
    print(f"Columns in DataFrame for main record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print(f"Record set {main_record_set_id} not loaded. Please verify @id.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section can include removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Example numeric field by its @id (e.g. coverage)
numeric_field_id = 'https://api.app.sen.science/frontiers/7154140/33e01dea-3893-4146-88e4-5c6e72dabaee'  # e.g. 'coverage_percentage'
# Example grouping field by its @id (e.g. a protein classification)
group_field_id = 'https://api.app.sen.science/frontiers/7154140/f5b3426c-17ca-4f34-98cf-d2008fda0493'    # e.g. 'description'

df = dataframes.get(main_record_set_id)
if df is not None and numeric_field_id in df.columns:
    # Remove rows with missing or non-numeric values
    df = df.copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    df = df.dropna(subset=[numeric_field_id])

    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    norm_col = numeric_field_id + '_normalized'
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped data by {group_field_id} (mean of numeric field):")
        print(grouped_df.head())
else:
    print(f"Could not process numeric analysis, check if {numeric_field_id} exists in DataFrame columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

df = dataframes.get(main_record_set_id)
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by group {group_field_id}")
        plt.xticks(rotation=90)
        plt.show()
else:
    print(f"Cannot plot: {numeric_field_id} not in DataFrame or main record set not loaded.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.